In [ ]:
from cascaqit import Circuit

circuit = Circuit(["q0", "q1"], program_id="program.digital.402a20e8-8437-4037-b969-33d934aae0f2")
circuit.h("q0")
circuit.cx("q0", "q1")
circuit.measure_all(key="m")
program = circuit.to_program()


In [ ]:
from cascaqit import AHSProgram, AtomRegister, MockNeutralAtomTarget, Waveform

layout_register = AtomRegister.square(
    side=3,
    spacing=5.0,
    origin=(-2.5, -5.0),
)
layout_positions = tuple(
    (round(site.position[0], 6), round(site.position[1], 6))
    for site in layout_register.sites[:9]
)
register = AtomRegister.custom(
    layout_positions,
    site_ids=["s0", "s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"],
    atom_ids=["s0", "s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"],
)

rabi = Waveform.piecewise_linear(
    times=[0.0, 0.4, 0.8, 1.2],
    values=[0.0, 2.5, 2.5, 0.0],
    waveform_id="rabi",
    value_unit="rad/us",
)
detuning = Waveform.piecewise_linear(
    times=[0.0, 1.2],
    values=[-4.0, 4.0],
    waveform_id="detuning",
    value_unit="rad/us",
)
phase = Waveform.piecewise_linear(
    times=[0.0, 1.2],
    values=[0.0, 0.0],
    waveform_id="phase",
    value_unit="rad",
)

builder = AHSProgram(register, program_id="program.analog.e157650b-dbbb-41ba-9e19-912a83f51e79")
builder.drive(rabi=rabi, detuning=detuning, phase=phase).measure()
program = builder.to_ir()
validation = builder.validate(MockNeutralAtomTarget.v0_1(), shots=100)
